# Supplementary Figure: motif-guided analog characterisation

Three panels, rendered as a single 6.5 x 8 in figure for Cell submission.

- **Panel A** - MSAs for cecropin, pa4, sarcotoxin, LG21 (LPS scaffolds) and bZIP (DNA scaffold).
  Template fully coloured by chemistry; analog residues coloured only where they match the
  template (conservation-on-template view). Template names and lead analogs are bold.
- **Panel B** - LPS BC-displacement at 32 uM, -Ca vs +Ca, one subplot per LPS family.
  Polymyxin B shown in every subplot as charge-sensitivity reference.
- **Panel C** - DNA deltaA heatmap for bZIP analogs across 12.5 / 25 / 50 uM.

All fontsizes are >= 6 pt.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle, Patch

PROJECT = Path("/home/pszymczak/code/omegamp-dashboard/data")
OUT = Path("/home/pszymczak/code/omegamp-dashboard/figures")
OUT.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 7,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.labelsize": 7,
    "axes.titlesize": 7.5,
    "legend.fontsize": 6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

## Colours and lead list

`LEAD_INDICES` defines which analogs are bolded in Panel A. Values come from
`table_selected_leads.csv` (LPS binding and DNA binding sections only - LG21
has no lead analog in that table).

In [ ]:
# Panel colours (matched to main Figure 6)
LPS_COLOR = "#4A90B8"
LPS_CA_COLOR = "#1E5A80"
DNA_COLOR = "#D4773A"
PROTO_COLOR = "#1E3A5F"
POLYMYX_COLOR = "#888888"

# ClustalX-style residue palette
AA_COLORS = {
    # Hydrophobic (blue)
    "A": "#87A7E0", "I": "#87A7E0", "L": "#87A7E0", "M": "#87A7E0",
    "F": "#87A7E0", "W": "#87A7E0", "V": "#87A7E0", "C": "#E787E0",
    # Cationic (red)
    "K": "#E87A7A", "R": "#E87A7A",
    # Anionic (magenta)
    "D": "#C256B0", "E": "#C256B0",
    # Polar (green)
    "N": "#85C285", "Q": "#85C285", "S": "#85C285", "T": "#85C285",
    # Glycine (orange), Proline (yellow)
    "G": "#F0A555", "P": "#D4C540",
    # His / Tyr (cyan)
    "H": "#5EC6C6", "Y": "#5EC6C6",
}

# Lead analogs per family (index matches Omega-N numbering)
LEAD_INDICES = {
    "cecropin":   {1, 4},
    "pa4":        {1},
    "sarcotoxin": {4},
    "bZIP":       {8},
    "LG21":       set(),
}

## Load data

In [ ]:
ref = pd.read_csv(PROJECT / "omegamp_reference_table.csv")
lps = pd.read_csv(PROJECT / "lps_binding.csv")
dna = pd.read_csv(PROJECT / "dna_binding.csv")

print(f"reference rows: {len(ref)}")
print(f"LPS rows:       {len(lps)}")
print(f"DNA rows:       {len(dna)}")

## Sequence lookup

`get_family_sequences` returns `(labels, seqs, bolds)`:

- `labels[0]` is the template name (e.g. `cecropin`); `labels[1..10]` are `Ω-1`..`Ω-10`.
- `bolds[i]` is True for the template row and for any analog whose index is in `LEAD_INDICES`.

In [ ]:
def get_family_sequences(ref, template_name):
    tpl = ref[ref["short_name"] == template_name].iloc[0]
    labels = [template_name]
    seqs = [tpl["sequence"]]
    bolds = [True]

    leads = LEAD_INDICES.get(template_name, set())
    for i in range(1, 11):
        analog = f"\u03a9-AMT-{template_name}-{i}"
        row = ref[ref["short_name"] == analog]
        if len(row):
            labels.append(f"\u03a9-{i}")
            seqs.append(row.iloc[0]["sequence"])
            bolds.append(i in leads)
    return labels, seqs, bolds


# Quick check
labels, seqs, bolds = get_family_sequences(ref, "cecropin")
for lab, seq, b in zip(labels, seqs, bolds):
    marker = "*" if b else " "
    print(f"{marker} {lab:8s} {seq}")

## Panel A: alignment drawing

Template row gets full chemistry colouring. Each analog residue is coloured only where
it matches the template at that position; everywhere else it's white. Mutations read
as white cells against the coloured background.

In [ ]:
def draw_alignment(ax, labels, seqs, bolds, title, fontsize=6):
    n = len(seqs)
    max_len = max(len(s) for s in seqs)
    label_space = 7.0
    template = seqs[0]

    for i, (lab, seq) in enumerate(zip(labels, seqs)):
        y = n - 1 - i
        is_template = (i == 0)
        weight = "bold" if bolds[i] else "normal"

        ax.text(
            -0.5, y + 0.5, lab, ha="right", va="center",
            fontsize=fontsize, family="monospace", fontweight=weight,
        )
        for j, aa in enumerate(seq):
            if is_template:
                color = AA_COLORS.get(aa, "white")
            else:
                if j < len(template) and aa == template[j]:
                    color = AA_COLORS.get(aa, "white")
                else:
                    color = "white"
            ax.add_patch(Rectangle(
                (j, y), 1, 1, facecolor=color, edgecolor="none", linewidth=0,
            ))
            ax.text(
                j + 0.5, y + 0.5, aa, ha="center", va="center",
                fontsize=fontsize, family="monospace", color="black",
                fontweight=weight,
            )

    ax.text(
        max_len / 2, n + 0.35, title, ha="center", va="bottom",
        fontsize=fontsize + 1, fontweight="bold",
    )
    ax.set_xlim(-label_space, max_len + 0.3)
    ax.set_ylim(-0.3, n + 1.3)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)

## Panel B: LPS BC-displacement (-Ca vs +Ca)

Each subplot is one family. Per peptide: mean -Ca displacement on x-axis, mean +Ca
displacement on y-axis, error bars = SD of three replicates. Diagonal marks
"no calcium effect". Points above diagonal = Ca-tolerant / enhanced; points below =
Ca-suppressed (electrostatics-dependent). Polymyxin B is shown in every subplot as
a charge-sensitivity reference.

In [ ]:
def draw_lps_family(ax, lps, family, template_name, lead_name=None,
                    lead_label=None, axis_max=75):
    sub = lps[(lps["family"] == family) & (lps["concentration"] == 32)]
    summary = (sub.groupby(["short_name", "calcium"])["BC_displacement"]
               .agg(["mean", "std"]).reset_index())
    mean_pv = summary.pivot(index="short_name", columns="calcium", values="mean")
    std_pv = summary.pivot(index="short_name", columns="calcium", values="std")

    ax.plot([0, axis_max], [0, axis_max], "--", color="#888888",
            linewidth=0.5, zorder=1)
    # Regime labels on the LG21 subplot only (cleanest background)
    if family == "LG21":
        ax.text(axis_max * 0.04, axis_max * 0.95, "Ca-tolerant", fontsize=6,
                color="#555555", ha="left", va="top", style="italic")
        ax.text(axis_max * 0.96, axis_max * 0.06, "Ca-suppressed", fontsize=6,
                color="#555555", ha="right", va="bottom", style="italic")

    lead_xy = None
    for name in mean_pv.index:
        if "no" not in mean_pv.columns or "yes" not in mean_pv.columns:
            continue
        x = mean_pv.loc[name, "no"]
        y = mean_pv.loc[name, "yes"]
        xerr = std_pv.loc[name, "no"] if not pd.isna(std_pv.loc[name, "no"]) else 0
        yerr = std_pv.loc[name, "yes"] if not pd.isna(std_pv.loc[name, "yes"]) else 0
        if pd.isna(x) or pd.isna(y):
            continue

        if name == template_name:
            marker, color, s, z, ec = "D", PROTO_COLOR, 32, 10, "white"
        elif name == "Polymyxin B":
            marker, color, s, z, ec = "s", "white", 22, 9, POLYMYX_COLOR
        elif lead_name and name == lead_name:
            marker, color, s, z, ec = "o", LPS_COLOR, 28, 8, "black"
            lead_xy = (x, y)
        else:
            marker, color, s, z, ec = "o", LPS_COLOR, 14, 5, "white"

        ax.errorbar(
            x, y, xerr=xerr, yerr=yerr, fmt="none",
            ecolor=color if name != "Polymyxin B" else POLYMYX_COLOR,
            elinewidth=0.4, alpha=0.45, zorder=z - 0.5,
        )
        ax.scatter(
            x, y, marker=marker, c=color, s=s, edgecolors=ec,
            linewidths=0.5, zorder=z,
        )

    # Lead annotation with smart offset
    if lead_xy and lead_label:
        lx, ly = lead_xy
        if lx > axis_max * 0.6 and ly > axis_max * 0.6:
            dx, dy = -20, 8
        elif lx > axis_max * 0.6:
            dx, dy = -22, 10
        else:
            dx, dy = 14, 12
        ax.annotate(
            lead_label, xy=(lx, ly), xytext=(lx + dx, ly + dy),
            fontsize=6, ha="center", fontweight="bold",
            arrowprops=dict(arrowstyle="-", color="black", lw=0.4,
                            shrinkA=0, shrinkB=2),
        )

    ax.set_xlim(0, axis_max)
    ax.set_ylim(0, axis_max)
    ax.set_aspect("equal")
    ax.set_title(family, fontsize=7, fontweight="bold", pad=2)
    ax.tick_params(length=2)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

## Panel C: DNA deltaA heatmap

Rows: bZIP template first, then analogs ordered by max |deltaA| descending.
Columns: 12.5 / 25 / 50 uM. Diverging colormap centred at 0 so bZIP's -5.8 at
25 uM reads as blue while positive perturbations read as red.

In [ ]:
def draw_dna_panel(ax, dna, cax):
    d = dna[dna["metric"] == "deltaA"].copy()
    pv = d.pivot(index="short_name", columns="concentration", values="value")

    template = "bZIP"
    analogs = [n for n in pv.index if n != template]
    analogs_sorted = sorted(analogs, key=lambda n: pv.loc[n].abs().max(),
                            reverse=True)
    pv = pv.loc[[template] + analogs_sorted]

    def short(n):
        if n == "bZIP":
            return "bZIP"
        return f"\u03a9-{n.split('-')[-1]}"

    y_labels = [short(n) for n in pv.index]

    vmax = max(abs(pv.values.min()), abs(pv.values.max()))
    im = ax.imshow(pv.values, cmap="RdBu_r", aspect="auto",
                   vmin=-vmax, vmax=vmax)

    for i in range(pv.shape[0]):
        for j in range(pv.shape[1]):
            val = pv.iloc[i, j]
            if pd.isna(val):
                continue
            t_color = "white" if abs(val) > vmax * 0.6 else "black"
            ax.text(j, i, f"{val:.1f}", ha="center", va="center",
                    fontsize=6, color=t_color)

    ax.set_xticks(range(len(pv.columns)))
    ax.set_xticklabels([f"{c:g}" for c in pv.columns], fontsize=6)
    ax.set_yticks(range(len(pv.index)))
    ax.set_yticklabels(y_labels, fontsize=6)

    bold_set = {"bZIP", "\u03a9-8", "\u03a9-9", "\u03a9-2"}
    for tick in ax.get_yticklabels():
        if tick.get_text() in bold_set:
            tick.set_fontweight("bold")

    ax.set_xlabel("Concentration (\u03bcmol L$^{-1}$)", fontsize=6.5, labelpad=2)
    ax.tick_params(length=2)
    for side in ("top", "right", "bottom", "left"):
        ax.spines[side].set_visible(False)

    cbar = plt.colorbar(im, cax=cax)
    cbar.set_label("\u0394A (mdeg)", fontsize=6.5, labelpad=2)
    cbar.ax.tick_params(labelsize=6, length=2)
    cbar.outline.set_linewidth(0.4)

## Assemble the full figure

Layout:

- Panel A spans the top, using 3 centered rows (cecropin+pa4, sarcotoxin+LG21, bZIP alone).
- Panel B: 1 row x 4 LPS subplots.
- Panel C: DNA heatmap with colorbar.

Panel letters (A / B / C) are positioned at the figure edge above each block.

In [ ]:
fig = plt.figure(figsize=(6.5, 8.0))

char_w = 0.078
fig_w, fig_h = 6.5, 8.0
fw = char_w / fig_w
fh = char_w / fig_h
label_cells = 7.0
align_rows = 12
row_h = align_rows * fh + 0.008

# ---------------- Panel A (LPS scatter, top) ----------------
panel_a_top = 0.985
panel_a_bot = 0.72
panel_a_h = panel_a_top - panel_a_bot
b_left, b_right = 0.075, 0.97
b_width = b_right - b_left
sub_gap = 0.025
sub_w = (b_width - 3 * sub_gap) / 4
sub_h = panel_a_h - 0.06
b_y = panel_a_bot + 0.01

families = [
    ("cecropin",   "cecropin",   "\u03a9-AMT-cecropin-1",   "\u03a9-1"),
    ("LG21",       "LG21",       None,                       None),
    ("pa4",        "pa4",        "\u03a9-AMT-pa4-1",        "\u03a9-1"),
    ("sarcotoxin", "sarcotoxin", "\u03a9-AMT-sarcotoxin-4", "\u03a9-4"),
]

axes_a = []
for i, (family, template, lead_name, lead_label) in enumerate(families):
    ax = fig.add_axes([b_left + i * (sub_w + sub_gap), b_y, sub_w, sub_h])
    draw_lps_family(ax, lps, family, template,
                    lead_name=lead_name, lead_label=lead_label, axis_max=75)
    if i == 0:
        ax.set_ylabel("+Ca$^{2+}$ BC displ. (%)", fontsize=6.5)
    else:
        ax.set_yticklabels([])
    ax.set_xlabel("-Ca$^{2+}$ BC displ. (%)", fontsize=6.5)
    axes_a.append(ax)

fig.text(0.005, panel_a_top, "A", fontsize=10, fontweight="bold", va="top")

for lab, fc, mk, ec in [
    ("Template",    PROTO_COLOR, "D", "white"),
    ("Analog",      LPS_COLOR,   "o", "white"),
    ("Polymyxin B", "white",     "s", POLYMYX_COLOR),
]:
    axes_a[0].scatter([], [], marker=mk, c=fc, edgecolors=ec,
                      s=25, linewidths=0.5, label=lab)
axes_a[0].legend(
    loc="upper left", bbox_to_anchor=(-0.15, 1.30), frameon=False,
    fontsize=6, handletextpad=0.2, columnspacing=0.6, ncol=3, borderpad=0,
)

# ---------------- Panel B (Sequence alignments, middle) ----------------
align_top = 0.695
gap = 0.03

y1 = align_top - row_h
wA_cec = (label_cells + 31 + 0.6) * fw
wA_pa4 = (label_cells + 33 + 0.6) * fw
row1_left = (1.0 - (wA_cec + gap + wA_pa4)) / 2
ax_cec = fig.add_axes([row1_left, y1, wA_cec, row_h])
ax_pa4 = fig.add_axes([row1_left + wA_cec + gap, y1, wA_pa4, row_h])

wA_sar = (label_cells + 39 + 0.6) * fw
wA_lg21 = (label_cells + 21 + 0.6) * fw
row2_left = (1.0 - (wA_sar + gap + wA_lg21)) / 2
y2 = y1 - row_h - 0.012
ax_sar = fig.add_axes([row2_left, y2, wA_sar, row_h])
ax_lg21 = fig.add_axes([row2_left + wA_sar + gap, y2, wA_lg21, row_h])

wA_bzip = (label_cells + 30 + 0.6) * fw
row3_left = (1.0 - wA_bzip) / 2
y3 = y2 - row_h - 0.012
ax_bzip = fig.add_axes([row3_left, y3, wA_bzip, row_h])

for template_name, ax in [
    ("cecropin", ax_cec), ("pa4", ax_pa4),
    ("sarcotoxin", ax_sar), ("LG21", ax_lg21),
    ("bZIP", ax_bzip),
]:
    labels, seqs, bolds = get_family_sequences(ref, template_name)
    draw_alignment(ax, labels, seqs, bolds, template_name, fontsize=6)

fig.text(0.008, align_top, "B", fontsize=10, fontweight="bold", va="top")

# AA chemistry legend below alignments
legend_y = y3 - 0.022
legend_items = [
    ("Cationic (K,R)", "#E87A7A"),
    ("Anionic (D,E)", "#C256B0"),
    ("Hydrophobic", "#87A7E0"),
    ("Polar (S,T,N,Q)", "#85C285"),
    ("Glycine", "#F0A555"),
    ("Proline", "#D4C540"),
    ("His/Tyr", "#5EC6C6"),
]
handles = [Patch(facecolor=c, edgecolor="none", label=l)
           for l, c in legend_items]
fig.legend(
    handles=handles, loc="upper center",
    bbox_to_anchor=(0.5, legend_y + 0.015),
    ncol=len(legend_items), fontsize=6, frameon=False,
    handletextpad=0.3, columnspacing=0.8,
    handlelength=0.9, handleheight=0.9, borderpad=0,
)

# ---------------- Panel C (DNA heatmap, bottom) ----------------
panel_c_top = legend_y - 0.018
panel_c_bot = 0.030
c_left, c_right = 0.22, 0.78
c_h = panel_c_top - panel_c_bot - 0.025
ax_dna = fig.add_axes([c_left, panel_c_bot + 0.015, c_right - c_left, c_h])
cax = fig.add_axes([c_right + 0.015, panel_c_bot + 0.015, 0.012, c_h])
draw_dna_panel(ax_dna, dna, cax)

fig.text(0.005, panel_c_top, "C", fontsize=10, fontweight="bold", va="top")
ax_dna.set_title("bZIP analogs", fontsize=7, fontweight="bold", pad=3)

plt.show()

## Save

In [ ]:
out_png = OUT / "figureS4_mechanism.png"
out_pdf = OUT / "figureS4_mechanism.pdf"
fig.savefig(out_png, dpi=400, bbox_inches=None)
fig.savefig(out_pdf, bbox_inches=None)
print(f"saved: {out_png}")
print(f"saved: {out_pdf}")